<a href="https://colab.research.google.com/github/JOk3r01001/mun/blob/main/kerbal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

path = '/content/drive/MyDrive/kerbal'



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os
import shutil

# Create a local directory to store the CSV files
local_csv_dir = './local_csv_files'
os.makedirs(local_csv_dir, exist_ok=True)

# List all files in the specified path
file_list = os.listdir(path)

print(f"Searching for CSV files in: {path}")
print(f"Files found: {file_list}")

csv_files_copied = []
for filename in file_list:
    if filename.endswith('.csv'):
        source_path = os.path.join(path, filename)
        destination_path = os.path.join(local_csv_dir, filename)
        shutil.copy(source_path, destination_path)
        csv_files_copied.append(filename)
        print(f"Copied '{filename}' to '{local_csv_dir}'")

if csv_files_copied:
    print(f"\nSuccessfully copied {len(csv_files_copied)} CSV files: {csv_files_copied}")
else:
    print("\nNo CSV files found or copied.")

Searching for CSV files in: /content/drive/MyDrive/kerbal
Files found: ['flight_1.csv', 'flight_2.csv', 'flight_3.csv', 'flight_4.csv', 'flight_5.csv', 'flight_6.csv', 'flight_7.csv']
Copied 'flight_1.csv' to './local_csv_files'
Copied 'flight_2.csv' to './local_csv_files'
Copied 'flight_3.csv' to './local_csv_files'
Copied 'flight_4.csv' to './local_csv_files'
Copied 'flight_5.csv' to './local_csv_files'
Copied 'flight_6.csv' to './local_csv_files'
Copied 'flight_7.csv' to './local_csv_files'

Successfully copied 7 CSV files: ['flight_1.csv', 'flight_2.csv', 'flight_3.csv', 'flight_4.csv', 'flight_5.csv', 'flight_6.csv', 'flight_7.csv']


In [8]:

import os
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

# 1. Load the Data directly from Drive
path = './local_csv_files'
all_files = glob.glob(os.path.join(path, "*.csv"))

if not all_files:
    print(f"Error: No CSV files found in {path}! Check the directory.")
    exit()

print("Fusing expert data...")
df_list = [pd.read_csv(f) for f in all_files]
df = pd.concat(df_list, ignore_index=True)

print(f"Loaded {len(df)} rows of 12-sensor expert flight data!")

# Extract Sensors (X) - NOW 12 SENSORS - and Actions (Y)
X_data = df[['obs_alt', 'obs_vel', 'obs_fuel', 'obs_pitch', 'obs_heading', 'obs_roll',
             'obs_pos_x', 'obs_pos_y', 'obs_pos_z', 'obs_ap', 'obs_pe', 'obs_time_to_ap']].values.astype(np.float32)
Y_data = df[['act_throttle', 'act_stage']].values.astype(np.float32)

# Calculate normalization statistics
X_mean = np.mean(X_data, axis=0)
X_std = np.std(X_data, axis=0)
X_std[X_std == 0] = 1e-8

# 2. Prepare Dataset
class KSPDataset(Dataset):
    def __init__(self, x, y):
        self.x = torch.tensor(x)
        self.y = torch.tensor(y)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

dataset = KSPDataset(X_data, Y_data)

# 80/20 Split
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# 3. PyTorch Brain Architecture
class KSPCloneBrain(nn.Module):
    def __init__(self, input_dim, output_dim, mean, std):
        super(KSPCloneBrain, self).__init__()

        # implements normalization logic directly into the .pth file
        self.register_buffer('x_mean', torch.tensor(mean, dtype=torch.float32))
        self.register_buffer('x_std', torch.tensor(std, dtype=torch.float32))

        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, output_dim),
            nn.Sigmoid()
        )

    def forward(self, x):
        x_norm = (x - self.x_mean) / self.x_std
        return self.network(x_norm)

# Initialize model with 12 input dimensions
model = KSPCloneBrain(input_dim=12, output_dim=2, mean=X_mean, std=X_std)
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
criterion = nn.MSELoss()

# 4. Training Loop
epochs = 75
print("\n>>> Commencing PyTorch Neural Network Training... <<<")

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        predictions = model(batch_x)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer


        optimizer.step()
        total_loss += loss.item()

    # --- EVALUATION BLOCK ---
    model.eval()
    test_loss = 0
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            preds = model(batch_x)
            test_loss += criterion(preds, batch_y).item()

    # Print progress every 10 epochs
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {total_loss/len(train_loader):.4f} | Test Loss: {test_loss/len(test_loader):.4f}")

# 5. Save the Finished model

save_folder = "/content/drive/MyDrive/kerbal"
save_path = os.path.join(save_folder, "ksp_expert_brain.pth")
torch.save(model.state_dict(), save_path)
print(f"\n>>> Brain completely trained and permanently saved to: {save_path} <<<")

Fusing expert data...
Loaded 12693 rows of 12-sensor expert flight data!

>>> Commencing PyTorch Neural Network Training... <<<
Epoch 1/75 | Train Loss: 0.0309 | Test Loss: 0.0109
Epoch 10/75 | Train Loss: 0.0070 | Test Loss: 0.0064
Epoch 20/75 | Train Loss: 0.0052 | Test Loss: 0.0046
Epoch 30/75 | Train Loss: 0.0048 | Test Loss: 0.0053
Epoch 40/75 | Train Loss: 0.0053 | Test Loss: 0.0045
Epoch 50/75 | Train Loss: 0.0044 | Test Loss: 0.0068
Epoch 60/75 | Train Loss: 0.0042 | Test Loss: 0.0051
Epoch 70/75 | Train Loss: 0.0039 | Test Loss: 0.0051

>>> Brain completely trained and permanently saved to: /content/drive/MyDrive/kerbal/ksp_expert_brain.pth <<<
